# Composition and Object Responsibilities


```{contents}
:local:
:depth: 2
```


```{index} Establish Required and Optional Relationships at Their Boundary
```



```{index} Establish Required and Optional Relationships at Their Boundary
```

## Establish Required and Optional Relationships at Their Boundary



An order line requires a product to calculate its amount. An order may optionally have a delivery note. Both are relationships, but absence has different meanings. Reject a missing required product when the line is created; allow an absent optional note only when callers know how that state is represented and displayed.

A nullable reference annotation such as `string?` communicates that absence is permitted for that reference. A nullable value type such as `int?` can distinguish an unknown quantity from a known zero. They are related ways of representing missing information, but nullable reference annotations do not wrap a runtime object or validate the constructor. Runtime guards remain necessary when a public contract rejects null.

| State | Meaning | Suitable check |
| --- | --- | --- |
| Product reference absent | Required relationship cannot be established | Reject at construction |
| Optional note absent | No note was supplied | Follow the documented no-note path |
| Note is empty text | An existing string contains no characters | Apply a separate text rule if needed |
| Quantity is zero | A known numeric value | Apply the business rule for zero |

Conditional access can skip a member access when a reference is absent; null coalescing can select a fallback value. Neither operator determines whether that fallback is a sensible business result. Returning zero for an unknown quantity may make arithmetic convenient while hiding incomplete data. State the policy first, then choose syntax to express it. The supporting nullability lesson retains the full operator and pattern variations.

Guided contract design: a customer can place an order without a note but never without an order line's product. List constructor outcomes and the expected total calculation for both relationships. Then check that a rejected construction or mutation leaves any already valid order state unchanged.

For related explanation, see the further reading below.[^development-reading]


A program that uses classes can still put all meaningful work in `Main`. Object-oriented design becomes useful when related objects take responsibility for their own behavior. An order-processing program gives you three distinct concepts: a product, a line requesting some quantity of that product, and an order containing lines.

Start from a use case: an employee adds two notebooks and three pens, then requests the order total. Name the responsibilities before writing fields. A class is justified by a coherent job, not simply by finding a noun in a problem statement.


```{figure} ../../figures/object_references.svg
:alt: Two variables alias one customer; an order owns private lines that refer to products.

References can be shared. Composition assigns responsibility for storing and protecting collaborators; it does not automatically clone every referenced object.
```


```{index} Has-a Relationships
```

## Has-a Relationships




```{index} Has-a Relationships
```


**Composition** builds a larger object from smaller objects. An order has lines; a line refers to a product. In contrast, an order is not a product. Inheritance would make the wrong promise in that relationship.

The term **aggregation** is sometimes used for a looser relationship in which an object refers to independently managed objects. In this design the order creates its lines, while several lines can share a product created by client code. State the actual ownership and sharing rules rather than relying on the terminology alone.

| Type | Responsibility | Information it needs |
| --- | --- | --- |
| Product | Describe one catalog version. | SKU, name, unit price. |
| OrderLine | Describe one requested quantity and calculate its amount. | Product reference and quantity. |
| Order | Maintain its lines and calculate the complete amount. | A private collection of order lines. |
| Console client | Accept input and display a result. | The public operations of the model. |

For the core example, repeated products create separate lines. There is no automatic merging or stock reservation. Those would be additional business rules with their own tests.


```{index} Let Each Object Calculate Its Own Result
```

(instance-methods)=

## Let Each Object Calculate Its Own Result




```{index} Let Each Object Calculate Its Own Result
```


`Product` protects its construction rules. `OrderLine` rejects a null product and nonpositive quantity before storing either value. Its `LineTotal` is calculated from the two values, so there is no second total field to become stale.

The example contains every class it needs. The program appears before the type declarations because these are ordinary top-level C# statements. Trace the property accesses from the client through the line to the product.


In [1]:
using System;
using System.Collections.Generic;

Product notebook = new Product("P100", "Notebook", 6m);
OrderLine requested = new OrderLine(notebook, 2);
Console.WriteLine($"{requested.Quantity} x {requested.Product.Name}: {requested.LineTotal:F2}");

class Product
{
    public string Sku { get; }
    public string Name { get; }
    public decimal UnitPrice { get; }

    public Product(string sku, string name, decimal unitPrice)
    {
        if (string.IsNullOrWhiteSpace(sku)) throw new ArgumentException("SKU required.");
        if (string.IsNullOrWhiteSpace(name)) throw new ArgumentException("Name required.");
        if (unitPrice < 0m) throw new ArgumentOutOfRangeException(nameof(unitPrice));
        Sku = sku.Trim();
        Name = name.Trim();
        UnitPrice = unitPrice;
    }
}

class OrderLine
{
    public Product Product { get; }
    public int Quantity { get; }
    public decimal LineTotal { get { return Product.UnitPrice * Quantity; } }

    public OrderLine(Product product, int quantity)
    {
        if (product is null) throw new ArgumentNullException(nameof(product));
        if (quantity <= 0) throw new ArgumentOutOfRangeException(nameof(quantity));
        Product = product;
        Quantity = quantity;
    }
}


2 x Notebook: 12.00


The line's constructor receives a reference to the product. It does not duplicate that product. Sharing is safe here because `Product` has no public state-changing operation. `OrderLine` is also fixed after construction. An order can therefore expose line objects without giving clients a way to rewrite their quantities.

If your design requires editable quantities, decide whether the order replaces a line or the line offers a validated operation. Do not add a setter merely because a property exists. The public surface should follow the use cases.


```{index} Keep Collection Ownership Inside the Order
```

## Keep Collection Ownership Inside the Order




```{index} Keep Collection Ownership Inside the Order
```


A private field is not protected if a public method hands the same mutable list to callers. A caller could clear the list, add a null, or bypass validation. The order should provide operations that preserve its rules instead.

`Add` creates and validates the line before inserting it. If construction fails, the existing list is unchanged. `Total` loops over current lines, starting at zero, so an empty order has a well-defined total. The private `readonly` field cannot be reassigned after initialization; its list can still change through `Add`. Readonly field binding is different from an immutable collection.


In [1]:
using System;
using System.Collections.Generic;

Product notebook = new Product("P100", "Notebook", 6m);
Product pen = new Product("P200", "Pen", 2m);
Order order = new Order();
order.Add(notebook, 2);
order.Add(pen, 3);
Console.WriteLine($"Lines: {order.Count}");
Console.WriteLine($"Total: {order.Total:F2}");

class Product
{
    public string Sku { get; }
    public string Name { get; }
    public decimal UnitPrice { get; }

    public Product(string sku, string name, decimal unitPrice)
    {
        if (string.IsNullOrWhiteSpace(sku)) throw new ArgumentException("SKU required.");
        if (string.IsNullOrWhiteSpace(name)) throw new ArgumentException("Name required.");
        if (unitPrice < 0m) throw new ArgumentOutOfRangeException(nameof(unitPrice));
        Sku = sku.Trim();
        Name = name.Trim();
        UnitPrice = unitPrice;
    }
}

class OrderLine
{
    public Product Product { get; }
    public int Quantity { get; }
    public decimal LineTotal { get { return Product.UnitPrice * Quantity; } }

    public OrderLine(Product product, int quantity)
    {
        if (product is null) throw new ArgumentNullException(nameof(product));
        if (quantity <= 0) throw new ArgumentOutOfRangeException(nameof(quantity));
        Product = product;
        Quantity = quantity;
    }
}

class Order
{
    private readonly List<OrderLine> lines = new List<OrderLine>();
    public int Count { get { return lines.Count; } }
    public decimal Total
    {
        get
        {
            decimal total = 0m;
            foreach (OrderLine line in lines) total += line.LineTotal;
            return total;
        }
    }
    public void Add(Product product, int quantity)
    {
        OrderLine line = new OrderLine(product, quantity);
        lines.Add(line);
    }
    public int CountUnits(string sku)
    {
        int count = 0;
        foreach (OrderLine line in lines)
            if (line.Product.Sku == sku) count += line.Quantity;
        return count;
    }
    public OrderLine[] GetLines() { return lines.ToArray(); }
}


Lines: 2
Total: 18.00


The output is two lines and a total of 18.00. `Main` coordinates the use case without calculating line amounts itself. Ask where each rule belongs: a negative unit price belongs to the product boundary; a positive quantity belongs to the line boundary; whether duplicate products merge belongs to the order's contract.

Avoid storing both `Total` and all the ingredients used to calculate it unless you also design how every update keeps them synchronized. Recalculation is a clear starting point for a small order. Later performance chapters give you tools to judge whether caching is justified.


```{index} Return Useful Results Without Exposing Storage
```

## Return Useful Results Without Exposing Storage




```{index} Return Useful Results Without Exposing Storage
```


`GetLines` returns a new array. Replacing an element in that array does not replace a line in the order's private list. The array is a shallow snapshot: its entries still refer to the same line objects. Because those lines and their products are immutable in this design, callers cannot use those references to change an existing line's amount.

A shallow copy would not provide the same protection with mutable line objects. Explain both the outer collection and its reachable elements when describing a snapshot.


In [1]:
using System;
using System.Collections.Generic;

Product notebook = new Product("P100", "Notebook", 6m);
Order order = new Order();
order.Add(notebook, 2);
OrderLine[] snapshot = order.GetLines();
snapshot[0] = new OrderLine(notebook, 99);
order.Add(notebook, 1);
Console.WriteLine($"Snapshot lines: {snapshot.Length}");
Console.WriteLine($"Current lines: {order.Count}");
Console.WriteLine($"Current total: {order.Total:F2}");

class Product
{
    public string Sku { get; }
    public string Name { get; }
    public decimal UnitPrice { get; }

    public Product(string sku, string name, decimal unitPrice)
    {
        if (string.IsNullOrWhiteSpace(sku)) throw new ArgumentException("SKU required.");
        if (string.IsNullOrWhiteSpace(name)) throw new ArgumentException("Name required.");
        if (unitPrice < 0m) throw new ArgumentOutOfRangeException(nameof(unitPrice));
        Sku = sku.Trim();
        Name = name.Trim();
        UnitPrice = unitPrice;
    }
}

class OrderLine
{
    public Product Product { get; }
    public int Quantity { get; }
    public decimal LineTotal { get { return Product.UnitPrice * Quantity; } }

    public OrderLine(Product product, int quantity)
    {
        if (product is null) throw new ArgumentNullException(nameof(product));
        if (quantity <= 0) throw new ArgumentOutOfRangeException(nameof(quantity));
        Product = product;
        Quantity = quantity;
    }
}

class Order
{
    private readonly List<OrderLine> lines = new List<OrderLine>();
    public int Count { get { return lines.Count; } }
    public decimal Total
    {
        get
        {
            decimal total = 0m;
            foreach (OrderLine line in lines) total += line.LineTotal;
            return total;
        }
    }
    public void Add(Product product, int quantity)
    {
        OrderLine line = new OrderLine(product, quantity);
        lines.Add(line);
    }
    public int CountUnits(string sku)
    {
        int count = 0;
        foreach (OrderLine line in lines)
            if (line.Product.Sku == sku) count += line.Quantity;
        return count;
    }
    public OrderLine[] GetLines() { return lines.ToArray(); }
}


Snapshot lines: 1
Current lines: 2
Current total: 18.00


The snapshot still has one entry, the order now has two, and the order total is 18.00. The snapshot replacement did not edit the order. This is a promise about the public API, not about preventing all possible code changes by the programmer who owns the source.

For a durable project, put `Product`, `OrderLine`, and `Order` in separate `.cs` files and keep the console entry point short. File organization should make responsibilities easy to locate; it does not change object relationships at runtime.


```{index} Practice: Count Requested Units
```

## Practice: Count Requested Units




```{index} Practice: Count Requested Units
```


Implement `CountUnits` by inspecting each line. Match SKU values using the chapter's case-sensitive policy. An unknown SKU returns zero; repeated lines contribute all their quantities.


In [ ]:
using System;
using System.Collections.Generic;

Product notebook = new Product("P100", "Notebook", 6m);
Order order = new Order();
order.Add(notebook, 2);
order.Add(notebook, 3);
Console.WriteLine(order.CountUnits("P100"));
Console.WriteLine(order.CountUnits("missing"));

class Product
{
    public string Sku { get; }
    public string Name { get; }
    public decimal UnitPrice { get; }

    public Product(string sku, string name, decimal unitPrice)
    {
        if (string.IsNullOrWhiteSpace(sku)) throw new ArgumentException("SKU required.");
        if (string.IsNullOrWhiteSpace(name)) throw new ArgumentException("Name required.");
        if (unitPrice < 0m) throw new ArgumentOutOfRangeException(nameof(unitPrice));
        Sku = sku.Trim();
        Name = name.Trim();
        UnitPrice = unitPrice;
    }
}

class OrderLine
{
    public Product Product { get; }
    public int Quantity { get; }
    public decimal LineTotal { get { return Product.UnitPrice * Quantity; } }

    public OrderLine(Product product, int quantity)
    {
        if (product is null) throw new ArgumentNullException(nameof(product));
        if (quantity <= 0) throw new ArgumentOutOfRangeException(nameof(quantity));
        Product = product;
        Quantity = quantity;
    }
}

class Order
{
    private readonly List<OrderLine> lines = new List<OrderLine>();
    public int Count { get { return lines.Count; } }
    public decimal Total
    {
        get
        {
            decimal total = 0m;
            foreach (OrderLine line in lines) total += line.LineTotal;
            return total;
        }
    }
    public void Add(Product product, int quantity)
    {
        OrderLine line = new OrderLine(product, quantity);
        lines.Add(line);
    }
    public int CountUnits(string sku)
    {
        int count = 0;
        foreach (OrderLine line in lines)
            // Your code starts here.
            { /* TODO: include matching quantities. */ }
            // Your code ends here.
        return count;
    }
    public OrderLine[] GetLines() { return lines.ToArray(); }
}


In [1]:
using System;
using System.Collections.Generic;

Product notebook = new Product("P100", "Notebook", 6m);
Order order = new Order();
order.Add(notebook, 2);
order.Add(notebook, 3);
Console.WriteLine(order.CountUnits("P100"));
Console.WriteLine(order.CountUnits("missing"));

class Product
{
    public string Sku { get; }
    public string Name { get; }
    public decimal UnitPrice { get; }

    public Product(string sku, string name, decimal unitPrice)
    {
        if (string.IsNullOrWhiteSpace(sku)) throw new ArgumentException("SKU required.");
        if (string.IsNullOrWhiteSpace(name)) throw new ArgumentException("Name required.");
        if (unitPrice < 0m) throw new ArgumentOutOfRangeException(nameof(unitPrice));
        Sku = sku.Trim();
        Name = name.Trim();
        UnitPrice = unitPrice;
    }
}

class OrderLine
{
    public Product Product { get; }
    public int Quantity { get; }
    public decimal LineTotal { get { return Product.UnitPrice * Quantity; } }

    public OrderLine(Product product, int quantity)
    {
        if (product is null) throw new ArgumentNullException(nameof(product));
        if (quantity <= 0) throw new ArgumentOutOfRangeException(nameof(quantity));
        Product = product;
        Quantity = quantity;
    }
}

class Order
{
    private readonly List<OrderLine> lines = new List<OrderLine>();
    public int Count { get { return lines.Count; } }
    public decimal Total
    {
        get
        {
            decimal total = 0m;
            foreach (OrderLine line in lines) total += line.LineTotal;
            return total;
        }
    }
    public void Add(Product product, int quantity)
    {
        OrderLine line = new OrderLine(product, quantity);
        lines.Add(line);
    }
    public int CountUnits(string sku)
    {
        int count = 0;
        foreach (OrderLine line in lines)
            if (line.Product.Sku == sku) count += line.Quantity;
        return count;
    }
    public OrderLine[] GetLines() { return lines.ToArray(); }
}


5
0


```{rubric} Footnotes
```


[^1102_composition-1]: Composition describes a has-a design. Garbage collection manages object lifetime through reachability; creating a line inside an order does not establish deterministic destruction or database ownership.


[^development-reading]: Further reading: [Microsoft, C# Record Types](https://learn.microsoft.com/en-us/dotnet/csharp/fundamentals/types/records). The worked business scenarios and guided reasoning here are original teaching examples.
